# 1M held-out pair test of the 800k-bal cross-encoder (model score only)

Tests `sagemaker_cross_800k_bal` (mDeBERTa-v3-base + LoRA) on **1,000,002 labelled pairs** that share
**no source1 entity** with the 800k training set or the earlier 500k test set. Matches are decided by the
model score alone: no fuzzy floor, and no blocking candidates.

| step | what | where |
|---|---|---|
| 1 | setup | this notebook |
| 2 | build the 1M test set (`sample_test_pairs.py`) and prove it has no overlap; skipped if it's already in S3 | this notebook (CPU, ~1 min) |
| 3 | score all pairs (`score_pairs.py`); skipped if all score shards already exist | SageMaker GPU jobs (~10 min) |
| 4 | metrics + charts (`eval_test_pairs.py`) | this notebook |

The test set uses random non-matches (the same kind the model was trained on), so expect scores near 1.0.
Realistic, look-alike candidates are much harder: see `sagemaker_infer_test.ipynb`.

Results from the first run (29 Sep 2026): ROC-AUC 0.99999996, pair F0.5 @0.5 = 0.99989 (29 false matches,
20 missed out of 1,000,002), per-S1 macro F0.5 = 0.99989.

## 1. Setup

In [ ]:
import glob, json, os, subprocess, tarfile, time
import boto3

REGION = 'ap-south-1'
BUCKET = 'sagemaker-ap-south-1-845989089398'
PREFIX = 'amazon-entity-resolution'
S3     = f's3://{BUCKET}/{PREFIX}'
ROLE   = 'arn:aws:iam::845989089398:role/service-role/AmazonSageMakerAdminIAMExecutionRole'
GPU_IMAGE  = '763104351884.dkr.ecr.ap-south-1.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker'
ADAPTER_S3 = f'{S3}/jobs/output/amazon-ml-cross-800k-bal-20260927-191710/output/model.tar.gz'

RUN_ID = 'test1m'                      # change to run a fresh copy
RUN    = f'{S3}/runs/{RUN_ID}'         # <RUN>/test/{pairs,scores}
N_PAIRS = 1_000_000
GPU_INSTANCES = ['ml.g5.xlarge', 'ml.g5.2xlarge', 'ml.g5.4xlarge']   # 1 A10G each; ~4 min for 1M pairs

ROOT = '/home/sagemaker-user/student_resource' if os.path.isdir('/home/sagemaker-user') else os.path.abspath('.')
CODE = f'{ROOT}/code/business_entity_resolution'
DATA = f'{ROOT}/dataset/train'
WORK = f'{ROOT}/work/{RUN_ID}'
for d in (DATA, WORK):
    os.makedirs(d, exist_ok=True)

sm, s3 = boto3.client('sagemaker', region_name=REGION), boto3.client('s3', region_name=REGION)

def sh(cmd):
    print('+', cmd); subprocess.run(cmd, shell=True, check=True)

def s3_count(prefix, suffix=''):
    b, _, k = prefix[5:].partition('/')
    return sum(o['Key'].endswith(suffix)
               for page in s3.get_paginator('list_objects_v2').paginate(Bucket=b, Prefix=k)
               for o in page.get('Contents', []))

sh(f'aws s3 sync {S3}/code/business_entity_resolution/ {CODE}/ --exclude "*__pycache__*" --only-show-errors')
sh('pip install -q polars==1.44.2 pyarrow matplotlib')

## 2. Build the 1M test set (or reuse the one in S3)

Whole source1 groups are sampled from `train_labeled_pairs.tsv` (natural 1 match : 3 non-match ratio),
skipping every source1 entity in the 800k training file **and** in the 500k test file. The overlap check
below must print 0 for both.

In [ ]:
PAIRS_DIR = f'{WORK}/pairs'
n_in_s3 = s3_count(f'{RUN}/test/pairs/', '.parquet')
if n_in_s3:
    print(f'reusing the {n_in_s3} test shards already in S3')
    sh(f'aws s3 sync {RUN}/test/pairs/ {PAIRS_DIR}/ --only-show-errors')
else:
    for local, remote in [(f'{DATA}/train_labeled_pairs.tsv', f'{S3}/dataset/train_labeled_pairs.tsv'),
                          (f'{DATA}/train_pairs_800k_balanced.tsv', f'{S3}/dataset/train_800k_bal/train_pairs_800k_balanced.tsv'),
                          (f'{DATA}/test_pairs_500k.tsv', f'{S3}/dataset/test500k/test_pairs_500k.tsv')]:
        if not os.path.exists(local):
            sh(f'aws s3 cp {remote} {local} --only-show-errors')
    sh(f'python {CODE}/src/sample_test_pairs.py --input {DATA}/train_labeled_pairs.tsv '
       f'--exclude {DATA}/train_pairs_800k_balanced.tsv {DATA}/test_pairs_500k.tsv '
       f'--out {DATA}/test_pairs_1m.tsv --shards {PAIRS_DIR} --n {N_PAIRS} --prefix test1m')
    sh(f'aws s3 sync {PAIRS_DIR}/ {RUN}/test/pairs/ --only-show-errors')
    sh(f'aws s3 cp {DATA}/test_pairs_1m.tsv {S3}/dataset/{RUN_ID}/test_pairs_1m.tsv --only-show-errors')

In [ ]:
# Overlap check: the test set must share no S1 entity (and no pair) with the training / 500k sets.
import polars as pl
for local, remote in [(f'{DATA}/train_pairs_800k_balanced.tsv', f'{S3}/dataset/train_800k_bal/train_pairs_800k_balanced.tsv'),
                      (f'{DATA}/test_pairs_500k.tsv', f'{S3}/dataset/test500k/test_pairs_500k.tsv')]:
    if not os.path.exists(local):
        sh(f'aws s3 cp {remote} {local} --only-show-errors')
te = pl.read_parquet(f'{PAIRS_DIR}/*.parquet', columns=['s1_id', 'cand_id', 'label'])
print(f'test set: {te.height:,} pairs, {te["label"].sum():,} matches, {te["s1_id"].n_unique():,} source1')
for name, path in [('800k training set', f'{DATA}/train_pairs_800k_balanced.tsv'),
                   ('500k test set', f'{DATA}/test_pairs_500k.tsv')]:
    o = (pl.scan_csv(path, separator='\t', quote_char=None, infer_schema=False)
         .select(pl.col('source1_entity_id').alias('s1_id'), pl.col('candidate_entity_id').alias('cand_id')).collect())
    s1_ov = te.join(o.select('s1_id').unique(), on='s1_id', how='semi')['s1_id'].n_unique()
    pair_ov = te.join(o, on=['s1_id', 'cand_id'], how='semi').height
    print(f'overlap with {name}: S1 entities={s1_ov}  pairs={pair_ov}')
    assert s1_ov == 0 and pair_ov == 0, 'test set overlaps!'

## 3. Score every pair on GPUs (skipped if already scored)

Each job runs `score_pairs.py`: one worker per GPU, all draining the same S3 shard queue, so any mix of
instances works and a job that can't get capacity is harmless. Scores are P(same business) from the model.

In [ ]:
n_pairs = s3_count(f'{RUN}/test/pairs/', '.parquet')
n_scored = s3_count(f'{RUN}/test/scores/', '.parquet')
print(f'shards: {n_pairs}, already scored: {n_scored}')

JOBS = []
if n_scored < n_pairs:
    bundle = '/tmp/test1m_sourcedir.tar.gz'
    with tarfile.open(bundle, 'w:gz') as t:
        t.add(f'{CODE}/sagemaker_job/infer_entry.py', arcname='infer_entry.py')
        t.add(f'{CODE}/sagemaker_job/requirements-infer.txt', arcname='requirements.txt')
        for f in ('lenient_blocking.py', 'blocking.py', 'fuzzy.py', 'normalize.py', 'phonetic.py', 'score_pairs.py'):
            t.add(f'{CODE}/src/{f}', arcname=f)
    SOURCE = f'{RUN}/source/sourcedir.tar.gz'
    sh(f'aws s3 cp {bundle} {SOURCE} --only-show-errors')
    stamp = time.strftime('%m%d-%H%M')
    for inst in GPU_INSTANCES:
        name = f"er-{RUN_ID}-score-{inst.split('.', 1)[1].replace('.', '-')}-{stamp}"[:63]
        try:
            sm.create_training_job(
                TrainingJobName=name, RoleArn=ROLE,
                AlgorithmSpecification={'TrainingImage': GPU_IMAGE, 'TrainingInputMode': 'File'},
                HyperParameters={'sagemaker_program': json.dumps('infer_entry.py'),
                                 'sagemaker_submit_directory': json.dumps(SOURCE),
                                 'sagemaker_region': json.dumps(REGION), 'mode': json.dumps('score'),
                                 'out-s3': json.dumps(f'{RUN}/test'), 'batch': json.dumps('256')},
                InputDataConfig=[{'ChannelName': 'adapter', 'DataSource': {'S3DataSource': {
                    'S3DataType': 'S3Prefix', 'S3Uri': ADAPTER_S3, 'S3DataDistributionType': 'FullyReplicated'}}}],
                OutputDataConfig={'S3OutputPath': f'{RUN}/jobs/'},
                ResourceConfig={'InstanceType': inst, 'InstanceCount': 1, 'VolumeSizeInGB': 30},
                StoppingCondition={'MaxRuntimeInSeconds': 2 * 3600},
                Environment={'HF_HUB_DISABLE_TELEMETRY': '1', 'TOKENIZERS_PARALLELISM': 'false'})
            JOBS.append(name); print('launched', name)
        except sm.exceptions.ClientError as e:
            print('could not launch', inst, '->', e)
else:
    print('all shards already scored: skipping GPU jobs')

In [ ]:
# Wait for scoring (re-run any time; interrupting does not stop the jobs).
while True:
    n_scored = s3_count(f'{RUN}/test/scores/', '.parquet')
    states = {j: sm.describe_training_job(TrainingJobName=j)['TrainingJobStatus'] for j in JOBS}
    print(time.strftime('%H:%M:%S'), f'scored {n_scored}/{n_pairs}', states, flush=True)
    if n_scored >= n_pairs:
        break
    if JOBS and all(s in ('Failed', 'Stopped', 'Completed') for s in states.values()):
        raise RuntimeError('all scoring jobs ended before every shard was scored')
    time.sleep(60)
sh(f'aws s3 sync {RUN}/test/scores/ {WORK}/scores/ --exclude "*.claim" --only-show-errors')

## 4. Metrics and charts (model score only)

In [ ]:
REPORT = f'{WORK}/report'
sh(f'cd {CODE}/src && python eval_test_pairs.py --pairs {PAIRS_DIR} --scores {WORK}/scores '
   f'--out {REPORT} --name 1M')
M = json.load(open(f'{REPORT}/metrics.json'))
sh(f'aws s3 sync {REPORT}/ {RUN}/report/ --only-show-errors')

In [ ]:
import pandas as pd
ts = M['test_set']
print(f"{ts['pairs']:,} pairs | {ts['matches']:,} matches | {ts['non_matches']:,} non-matches | {ts['source1_entities']:,} source1")
print(f"ROC-AUC {M['roc_auc']:.8f}   average precision {M['average_precision']:.8f}")
for t, v in M['source1_macro_f0_5'].items():
    print(f"per-S1 macro F0.5 @ {t}: {v['macro_f0_5']:.5f}  (no-match S1s correct: {v['singleton_accuracy']:.4%})")

pd.DataFrame(M['pair_threshold_table'])[['threshold', 'precision', 'recall', 'f0_5', 'tp', 'fp', 'fn', 'tn']] \
  .style.format({'precision': '{:.6f}', 'recall': '{:.6f}', 'f0_5': '{:.6f}'}).hide(axis='index')

In [ ]:
pd.DataFrame([{'country': c, **{k: v[k] for k in ('pairs', 'precision', 'recall', 'f0_5', 'fp', 'fn')}}
              for c, v in M['by_country_at_0.5'].items()]) \
  .style.format({'precision': '{:.5f}', 'recall': '{:.5f}', 'f0_5': '{:.5f}'}).hide(axis='index')

In [ ]:
from IPython.display import Image, display
for png in sorted(glob.glob(f'{REPORT}/*.png')):
    print(os.path.basename(png)); display(Image(filename=png, width=820))